In [2]:
%pip install "psycopg[binary]"

   ---------------------------------------- 0.0/3.7 MB ? eta -:--:--
   -- ------------------------------------- 0.3/3.7 MB ? eta -:--:--
   -------- ------------------------------- 0.8/3.7 MB 2.8 MB/s eta 0:00:02
   ----------------- ---------------------- 1.6/3.7 MB 3.5 MB/s eta 0:00:01
   ------------------------- -------------- 2.4/3.7 MB 3.6 MB/s eta 0:00:01
   ---------------------------------- ----- 3.1/3.7 MB 3.5 MB/s eta 0:00:01
   ---------------------------------------- 3.7/3.7 MB 3.3 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


In [1]:
import psycopg
import sqlalchemy

print("Psycopg version:", psycopg.__version__)
print("SQLAlchemy version:", sqlalchemy.__version__)

Psycopg version: 3.3.6
SQLAlchemy version: 2.0.51


In [2]:
#Connection to Postgre

from sqlalchemy import create_engine, URL, text
from getpass import getpass

db_url = URL.create(
    "postgresql+psycopg",
    username="postgres",
    password=getpass("Enter PostgreSQL password: "),
    host="localhost",
    port=5432,
    database="Car_Price"
)

engine = create_engine(db_url)

with engine.connect() as conn:
    result = conn.execute(
        text("SELECT current_database(), current_user")
    )
    print("Connected successfully:", result.fetchone())

Enter PostgreSQL password:  ········


Connected successfully: ('Car_Price', 'postgres')


In [9]:
import pandas as pd

df = pd.read_excel("Car_details_updated.xlsx")

df.head()

,name,year,selling_price,km_driven,fuel,seller_type,transmission,owner,mileage,engine,max_power,Torque,seats,RPM,Brand
0,Maruti Swift Dzire VDI,2014,450000,145500,Diesel,Individual,Manual,First Owner,23.40,1248.0,74.00,190.0,5.0,2000.00,Maruti
1,Skoda Rapid 1.5 TDI Ambition,2014,370000,120000,Diesel,Individual,Manual,Second Owner,21.14,1498.0,103.52,250.0,5.0,2500.00,Skoda
2,Honda City 2017-2020 EXi,2006,158000,140000,Petrol,Individual,Manual,Third Owner,17.70,1497.0,78.00,12.7,5.0,3058.59,Honda
3,Hyundai i20 Sportz Diesel,2010,225000,127000,Diesel,Individual,Manual,First Owner,23.00,1396.0,90.00,22.4,5.0,2750.00,Hyundai
4,Maruti Swift VXI BSIII,2007,130000,120000,Petrol,Individual,Manual,First Owner,16.10,1298.0,88.20,11.5,5.0,3058.59,Maruti


In [11]:
print(df.shape)
display(df.head())
print(df.dtypes)

(8128, 15)


,name,year,selling_price,km_driven,fuel,seller_type,transmission,owner,mileage,engine,max_power,Torque,seats,RPM,Brand
0,Maruti Swift Dzire VDI,2014,450000,145500,Diesel,Individual,Manual,First Owner,23.40,1248.0,74.00,190.0,5.0,2000.00,Maruti
1,Skoda Rapid 1.5 TDI Ambition,2014,370000,120000,Diesel,Individual,Manual,Second Owner,21.14,1498.0,103.52,250.0,5.0,2500.00,Skoda
2,Honda City 2017-2020 EXi,2006,158000,140000,Petrol,Individual,Manual,Third Owner,17.70,1497.0,78.00,12.7,5.0,3058.59,Honda
3,Hyundai i20 Sportz Diesel,2010,225000,127000,Diesel,Individual,Manual,First Owner,23.00,1396.0,90.00,22.4,5.0,2750.00,Hyundai
4,Maruti Swift VXI BSIII,2007,130000,120000,Petrol,Individual,Manual,First Owner,16.10,1298.0,88.20,11.5,5.0,3058.59,Maruti


name              object
year               int64
selling_price      int64
km_driven          int64
fuel              object
seller_type       object
transmission      object
owner             object
mileage          float64
engine           float64
max_power        float64
Torque           float64
seats            float64
RPM              float64
Brand             object
dtype: object


In [13]:
#Load the data into PostgreSQL
df.to_sql(
    name="car_price",
    con=engine,
    schema="public",
    if_exists="fail",
    index=False,
    chunksize=1000
)

print("Data loaded successfully!")

Data loaded successfully!


In [14]:
#verify the load
from sqlalchemy import text

with engine.connect() as conn:
    count = conn.execute(
        text("SELECT COUNT(*) FROM public.car_price")
    ).scalar_one()

    print("Rows in PostgreSQL:", count)

Rows in PostgreSQL: 8128


In [15]:
from sqlalchemy import text

with engine.connect() as conn:
    db_count = conn.execute(
        text("SELECT COUNT(*) FROM public.car_price")
    ).scalar_one()

print("DataFrame rows:", len(df))
print("Database rows:", db_count)
print("Row counts match:", len(df) == db_count)

DataFrame rows: 8128
Database rows: 8128
Row counts match: True
